week 4-5

In [344]:
import pandas as pd
df = pd.read_csv("CRMLSSold_with_rates.csv")
print(df.shape)  

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_4818/4012198578.py:2: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSSold_with_rates.csv")


(414184, 87)


Step 1  Process Date Fields

In [345]:

date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]
# Convert the date column to datetime.
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
print(df[date_cols].dtypes)
print(df[date_cols].head(5))

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object
   CloseDate PurchaseContractDate ListingContractDate ContractStatusChangeDate
0 2024-01-26           2023-11-22          2021-10-06               2024-01-26
1 2024-01-05           2021-06-30          2021-03-08               2024-01-05
2 2024-01-05           2021-11-18          2021-03-08               2024-01-05
3 2024-01-30           2024-08-05          2024-01-30               2024-01-30
4 2024-01-29           2024-01-29          2024-01-29               2024-01-29


In [346]:
# create date consistency flags

# A. listing date is after close date
df["listing_after_close_flag"] = (
    df["ListingContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["ListingContractDate"] > df["CloseDate"])
)

# B. purchase date is after close date
df["purchase_after_close_flag"] = (
    df["PurchaseContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["PurchaseContractDate"] > df["CloseDate"])
)

# C. negative timeline:
# listing date is after purchase date
df["negative_timeline_flag"] = (
    df["ListingContractDate"].notna() &
    df["PurchaseContractDate"].notna() &
    (df["ListingContractDate"] > df["PurchaseContractDate"])
)

#  check flag counts
print("Flag counts:")
print("listing_after_close_flag:", df["listing_after_close_flag"].sum())
print("purchase_after_close_flag:", df["purchase_after_close_flag"].sum())
print("negative_timeline_flag:", df["negative_timeline_flag"].sum())
print()

Flag counts:
listing_after_close_flag: 61
purchase_after_close_flag: 240
negative_timeline_flag: 270



In [347]:
print("Rows with listing_after_close_flag = 1")
print(df.loc[df["listing_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with purchase_after_close_flag = 1")
print(df.loc[df["purchase_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with negative_timeline_flag = 1")
print(df.loc[df["negative_timeline_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

Rows with listing_after_close_flag = 1
      ListingContractDate PurchaseContractDate  CloseDate
22             2024-01-31           2024-01-15 2024-01-30
83             2024-01-26           2024-01-10 2024-01-25
711            2024-01-02           2023-12-04 2024-01-01
24279          2024-04-08           2024-03-27 2024-03-29
24399          2024-03-21           2024-03-03 2024-03-20

Rows with purchase_after_close_flag = 1
   ListingContractDate PurchaseContractDate  CloseDate
3           2024-01-30           2024-08-05 2024-01-30
7           2024-01-31           2024-03-04 2024-01-31
11          2024-01-31           2024-02-05 2024-01-31
12          2024-01-29           2024-02-04 2024-01-29
20          2024-01-29           2024-02-01 2024-01-29

Rows with negative_timeline_flag = 1
    ListingContractDate PurchaseContractDate  CloseDate
22           2024-01-31           2024-01-15 2024-01-30
32           2024-01-18           2024-01-05 2024-01-30
48           2024-01-29           20

In [348]:
# High-Missing Columns (≥90%)
missing_pct = df.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct >= 90].index.tolist()
print("High-missing cols:", len(high_missing_cols))

# Non-analytical Column
extra_drop = [
    "BuyerAgentAOR", "ListAgentAOR",
    "ListAgentEmail",
    "ListAgentFirstName", "ListAgentLastName",
    "ListAgentFullName",
    "BuyerAgentFirstName", "BuyerAgentLastName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerOfficeAOR", "BuyerAgencyCompensationType", "BuyerAgencyCompensation"
]
extra_drop = [col for col in extra_drop if col in df.columns]
print("Manual drop cols:", len(extra_drop))

before_cols = df.shape[1]
# Execute Deletion
df = df.drop(columns=high_missing_cols)
df = df.drop(columns=extra_drop)

after_cols = df.shape[1]

print("===== Check Deletion =====")
print("Before:", before_cols)
print("After:", after_cols)
print("Removed:", before_cols - after_cols)

High-missing cols: 15
Manual drop cols: 13
===== Check Deletion =====
Before: 90
After: 62
Removed: 28


In [350]:
# ---------- Duplicate Column Check ----------
# 1. Suffix duplicates (.1, .2, etc.)
suffix_dups = [col for col in df.columns if col.endswith((".1", ".2", ".3"))]

# 2. Duplicate column names
name_dups = df.columns[df.columns.duplicated()].tolist()

# 3. Duplicate content columns
content_dups = []
cols = df.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df[cols[i]].equals(df[cols[j]]):
            content_dups.append((cols[i], cols[j]))

# ---------- Print Summary ----------
print("===== Duplicate Column Summary =====")
print(f"Suffix duplicates (.1 etc): {len(suffix_dups)}")
print(f"Duplicate column names: {len(name_dups)}")
print(f"Duplicate content pairs: {len(content_dups)}")

print("\n===== Duplicate Content Examples =====")

for col1, col2 in content_dups:
    print(f"\nComparing: {col1} vs {col2}")
    print(df[[col1, col2]].head(5))

===== Duplicate Column Summary =====
Suffix duplicates (.1 etc): 0
Duplicate column names: 0
Duplicate content pairs: 2

===== Duplicate Content Examples =====

Comparing: ListingKey vs ListingKeyNumeric
   ListingKey  ListingKeyNumeric
0   551985747          551985747
1   522107581          522107581
2   510919001          510919001
3  1079166779         1079166779
4  1075037759         1075037759

Comparing: latfilled vs lonfilled
  latfilled lonfilled
0       NaN       NaN
1       NaN       NaN
2       NaN       NaN
3       NaN       NaN
4       NaN       NaN


In [351]:

#---------- Drop Duplicate Columns ----------
df = df.drop(columns=["ListingKeyNumeric"])
print("Dropped ListingKeyNumeric (duplicate of ListingKey)")

# Remove duplicate rows 
# =========================
before = len(df)
df = df.drop_duplicates()
after = len(df)

print("\n[Sold]")
print(f"Before : {before:,} rows")
print(f"After  : {after:,} rows")
print(f"Removed: {before - after:,} duplicate rows")
print(df[["latfilled", "lonfilled"]].isnull().mean())
#Although latfilled and lonfilled appeared identical in some cases, further
# inspection showed that they primarily consist of missing values and represent data 
# quality flags rather than true duplicate variables. Therefore, they were not removed
#  as duplicate columns.

Dropped ListingKeyNumeric (duplicate of ListingKey)

[Sold]
Before : 414,184 rows
After  : 414,136 rows
Removed: 48 duplicate rows
latfilled    0.845741
lonfilled    0.845741
dtype: float64


In [352]:
#Check to ensure that the data type is correct
num_cols = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger"
]

print(df[num_cols].dtypes)
for col in num_cols:
    print(f"\n--- {col} sample values ---")
    print(df[col].head(2))
for col in num_cols:
    print(f"{col} missing after conversion:", df[col].isna().sum())

ClosePrice               float64
ListPrice                float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object

--- ClosePrice sample values ---
0    240000.0
1    815000.0
Name: ClosePrice, dtype: float64

--- ListPrice sample values ---
0    295000.0
1    759900.0
Name: ListPrice, dtype: float64

--- LivingArea sample values ---
0    1140.0
1    1974.0
Name: LivingArea, dtype: float64

--- DaysOnMarket sample values ---
0    777
1     33
Name: DaysOnMarket, dtype: int64

--- BedroomsTotal sample values ---
0    2.0
1    4.0
Name: BedroomsTotal, dtype: float64

--- BathroomsTotalInteger sample values ---
0    2.0
1    4.0
Name: BathroomsTotalInteger, dtype: float64
ClosePrice missing after conversion: 2
ListPrice missing after conversion: 0
LivingArea missing after conversion: 234
DaysOnMarket missing after conversion: 0
BedroomsTotal missing after conversion: 11
BathroomsTotalInteger 

In [353]:
# numeric columns
numeric_cols = [
    "ClosePrice",
    "OriginalListPrice",
    "ListPrice",
    "LivingArea",
    "LotSizeAcres",
    "DaysOnMarket",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "Latitude",
    "Longitude",
    "AssociationFee",
    "ParkingTotal",
    "GarageSpaces",
    "LotSizeSquareFeet",
    "rate_30yr_fixed"
]
# convert to numeric
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
# check data types
print("\nNumeric Column Types:")
print(df[numeric_cols].dtypes)

# preview
print("\nNumeric Columns Preview:")
print(df[numeric_cols].head(3))


Numeric Column Types:
ClosePrice               float64
OriginalListPrice        float64
ListPrice                float64
LivingArea               float64
LotSizeAcres             float64
DaysOnMarket               int64
YearBuilt                float64
BathroomsTotalInteger    float64
BedroomsTotal            float64
Latitude                 float64
Longitude                float64
AssociationFee           float64
ParkingTotal             float64
GarageSpaces             float64
LotSizeSquareFeet        float64
rate_30yr_fixed          float64
dtype: object

Numeric Columns Preview:
   ClosePrice  OriginalListPrice  ListPrice  LivingArea  LotSizeAcres  \
0    240000.0           499000.0   295000.0      1140.0           NaN   
1    815000.0           759900.0   759900.0      1974.0           NaN   
2    810000.0           739900.0   770000.0      1974.0           NaN   

   DaysOnMarket  YearBuilt  BathroomsTotalInteger  BedroomsTotal  Latitude  \
0           777     1988.0            

Step 3: Outlier Check (Invalid Values)

In [354]:
# -------------------------------
# Invalid value flags 
# -------------------------------

df["invalid_close_price_flag"] = df["ClosePrice"] <= 0
df["invalid_living_area_flag"] = df["LivingArea"] <= 0
df["invalid_days_on_market_flag"] = df["DaysOnMarket"] < 0
df["invalid_bedrooms_flag"] = df["BedroomsTotal"] < 0
df["invalid_bathrooms_flag"] = df["BathroomsTotalInteger"] < 0

# count
print("Invalid Value Counts:")
print(df[
    [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag",
        "invalid_bedrooms_flag",
        "invalid_bathrooms_flag"
    ]
].sum())

Invalid Value Counts:
invalid_close_price_flag         1
invalid_living_area_flag       150
invalid_days_on_market_flag     47
invalid_bedrooms_flag            0
invalid_bathrooms_flag           0
dtype: int64


Step 4：Geographic Data Checks

In [355]:
# -------------------------------
# Geographic flags
# -------------------------------

# 1. Missing Values
df["missing_geo_flag"] = (
    df["Latitude"].isna() | df["Longitude"].isna()
)

# 2. Zero Values (Dummy Data)
df["zero_geo_flag"] = (
    (df["Latitude"] == 0) | (df["Longitude"] == 0)
)

# 3. Incorrect longitude direction
# California longitude should be negative
df["invalid_longitude_flag"] = (
    df["Longitude"] > 0
)

# 4. Outside California range
df["out_of_ca_flag"] = (
    (df["Latitude"] < 32) | (df["Latitude"] > 42) |
    (df["Longitude"] < -125) | (df["Longitude"] > -114)
)

# Count issues
print("Geographic Issues Count:")
print("missing_geo_flag:", df["missing_geo_flag"].sum())
print("zero_geo_flag:", df["zero_geo_flag"].sum())
print("invalid_longitude_flag:", df["invalid_longitude_flag"].sum())
print("out_of_ca_flag:", df["out_of_ca_flag"].sum())

Geographic Issues Count:
missing_geo_flag: 15956
zero_geo_flag: 27
invalid_longitude_flag: 31
out_of_ca_flag: 88


In [356]:
# -------------------------------
# Clean data: numeric + date + geo
# -------------------------------

before_rows = len(df)

df = df[
    # numeric validity
    (df["ClosePrice"] > 0) &
    (df["LivingArea"] > 0) &
    (df["DaysOnMarket"] >= 0) &
    (
        (df["BedroomsTotal"].isna()) |
        (df["BedroomsTotal"] >= 0)
    ) &
    (
        (df["BathroomsTotalInteger"].isna()) |
        (df["BathroomsTotalInteger"] >= 0)
    ) &

    # date validity
    (df["listing_after_close_flag"] == 0) &
    (df["purchase_after_close_flag"] == 0) &
    (df["negative_timeline_flag"] == 0) &

    # geo validity
    (df["zero_geo_flag"] == 0) &
    (df["invalid_longitude_flag"] == 0) &
    (df["out_of_ca_flag"] == 0)
].copy()

after_rows = len(df_clean)

print("\nFinal Cleaning Summary:")
print("Rows before cleaning:", before_rows)
print("Rows after cleaning :", after_rows)
print("Rows removed        :", before_rows - after_rows)


Final Cleaning Summary:
Rows before cleaning: 414136
Rows after cleaning : 413106
Rows removed        : 1030


In [357]:
flag_cols = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_bedrooms_flag",
    "invalid_bathrooms_flag",
    "missing_geo_flag",
    "zero_geo_flag",
    "invalid_longitude_flag",
    "out_of_ca_flag"
]

df[flag_cols].sum().sort_values(ascending=False)

missing_geo_flag               15869
listing_after_close_flag           0
purchase_after_close_flag          0
negative_timeline_flag             0
invalid_close_price_flag           0
invalid_living_area_flag           0
invalid_days_on_market_flag        0
invalid_bedrooms_flag              0
invalid_bathrooms_flag             0
zero_geo_flag                      0
invalid_longitude_flag             0
out_of_ca_flag                     0
dtype: int64

week 6

In [ ]:
import pandas as pd
import numpy as np

# Key Metrics to Create
# =============================

# 1. Price Ratio
df["price_ratio"] = np.where(
    df["OriginalListPrice"] > 0,
    df["ClosePrice"] / df["OriginalListPrice"],
    np.nan
)

# 2. Close-to-Original-List Ratio
df["close_to_original_list_ratio"] = np.where(
    df["OriginalListPrice"] > 0,
    df["ClosePrice"] / df["OriginalListPrice"],
    np.nan
)

# 3. Price Per Square Foot
df["price_per_sqft"] = np.where(
    df["LivingArea"] > 0,
    df["ClosePrice"] / df["LivingArea"],
    np.nan
)

# 4. Year / Month / YrMo
df["close_year"] = df["CloseDate"].dt.year
df["close_month"] = df["CloseDate"].dt.month
df["YrMo"] = df["CloseDate"].dt.to_period("M").astype(str)

# 5. Listing-to-Contract Days
df["listing_to_contract_days"] = (
    df["PurchaseContractDate"] - df["ListingContractDate"]
).dt.days

# 6. Contract-to-Close Days
df["contract_to_close_days"] = (
    df["CloseDate"] - df["PurchaseContractDate"]
).dt.days

# Sample output table
# =============================

sample_cols = [
    "CloseDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "DaysOnMarket",
    "YrMo",
    "listing_to_contract_days",
    "contract_to_close_days"
]

print("\nSample engineered metrics:")
print(df[sample_cols].head(5))


# Segment summary table
# Grouped by CountyOrParish
# =============================

county_summary = (
    df.groupby("CountyOrParish")
    .agg(
        total_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("DaysOnMarket", "mean"),
        median_price_ratio=("price_ratio", "median"),
        average_listing_to_contract_days=("listing_to_contract_days", "mean"),
        average_contract_to_close_days=("contract_to_close_days", "mean")
    )
    .reset_index()
    .sort_values(by="median_close_price", ascending=False)
)

print("\nCounty segment summary:")
print(county_summary.head(10))

# =============================
# Save outputs
# =============================

df.to_csv("CRMLSSold_Week6_Features.csv", index=False)
county_summary.to_csv("county_segment_summary.csv", index=False)
print("Saved: CRMLSSold_Week6_Features.csv")
print("Saved: county_segment_summary.csv")


Sample engineered metrics:
   CloseDate  ClosePrice  OriginalListPrice  LivingArea  price_ratio  \
0 2024-01-26    240000.0           499000.0      1140.0     0.480962   
1 2024-01-05    815000.0           759900.0      1974.0     1.072510   
2 2024-01-05    810000.0           739900.0      1974.0     1.094743   
4 2024-01-29   1890500.0          1890500.0      3194.0     1.000000   
5 2024-01-02   2100000.0          2100000.0      3736.0     1.000000   

   close_to_original_list_ratio  price_per_sqft  DaysOnMarket     YrMo  \
0                      0.480962      210.526316           777  2024-01   
1                      1.072510      412.867275            33  2024-01   
2                      1.094743      410.334347           228  2024-01   
4                      1.000000      591.891046             0  2024-01   
5                      1.000000      562.098501             0  2024-01   

   listing_to_contract_days  contract_to_close_days  
0                     777.0             

In [ ]:
pd.read_csv("Sold_county_segment_summary.csv")

,CountyOrParish,total_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,median_price_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Del Norte,1,2485000.0,2.485000e+06,390.907661,320.000000,0.764615,320.000000,9.000000
1,San Mateo,7209,1700000.0,2.197712e+06,1050.000000,28.537245,1.013112,28.219448,24.453322
2,Santa Clara,18363,1600000.0,1.921440e+06,965.986395,21.968850,1.024016,22.113544,26.663236
3,Santa Cruz,2975,1200000.0,1.351482e+06,736.281736,38.960672,0.982906,39.016134,27.273613
4,San Francisco,949,1188888.0,1.318708e+06,886.363636,40.955743,1.001760,41.468915,26.884089
...,...,...,...,...,...,...,...,...,...
58,Imperial,289,306000.0,3.118750e+05,211.171662,55.664360,0.984440,73.041812,29.522648
59,Modoc,2,282000.0,2.820000e+05,104.858398,46.500000,0.985673,50.500000,41.500000
60,Sierra,1,255000.0,2.550000e+05,222.902098,1.000000,0.944444,47.000000,38.000000
61,Foreign Country,1,250000.0,2.500000e+05,106.837607,501.000000,0.781250,519.000000,4.000000


In [374]:
required_cols = [
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "DaysOnMarket",
    "CountyOrParish"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print("Missing columns:", missing_cols)
else:
    print("All required columns exist.")

All required columns exist.
